# 🎬 CineMatch — Film Recommendation System with SVD

**Capstone Project | Pijak × IBM SkillsBuild**  
**Tema:** Machine Learning / AI — Sistem Rekomendasi  
**Algoritma:** Model-Based Collaborative Filtering (Singular Value Decomposition)

---

## 📋 Daftar Isi

1. [Import Library](#1-import-library)
2. [Load Dataset](#2-load-dataset)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis)
4. [Preprocessing](#4-preprocessing)
5. [Training Model SVD](#5-training-model-svd)
6. [Evaluasi Model](#6-evaluasi-model)
7. [Prediksi & Rekomendasi](#7-prediksi--rekomendasi)
8. [Kesimpulan](#8-kesimpulan)

---
## 1. Import Library

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Sistem Rekomendasi
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV

# Utilitas
import pickle
import warnings
warnings.filterwarnings('ignore')

# Style plot
plt.style.use('dark_background')
ACCENT  = '#e8c547'
ACCENT2 = '#ff6b35'
SURFACE = '#1a1a26'
sns.set_palette([ACCENT, ACCENT2, '#7ec8e3', '#b5ead7', '#ff9aa2'])

print('✅ Semua library berhasil diimport')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

---
## 2. Load Dataset

Dataset yang digunakan adalah **MovieLens Latest Small** dari [GroupLens](https://grouplens.org/datasets/movielens/), yang merupakan dataset benchmark standar industri untuk sistem rekomendasi film.

| File | Keterangan |
|---|---|
| `ratings.csv` | Data rating user terhadap film |
| `movies.csv` | Metadata film (judul, genre) |

In [ ]:
# Load dataset
# Jika menjalankan dari folder cinematch/, gunakan path berikut:
ratings = pd.read_csv('ratings_data.csv')
movies  = pd.read_csv('movies_data.csv')

# Jika menggunakan dataset asli MovieLens:
# ratings = pd.read_csv('ml-latest-small/ratings.csv')
# movies  = pd.read_csv('ml-latest-small/movies.csv')

print('=== RATINGS ===')
print(f'Shape  : {ratings.shape}')
print(f'Kolom  : {list(ratings.columns)}')
display(ratings.head())

print('\n=== MOVIES ===')
print(f'Shape  : {movies.shape}')
print(f'Kolom  : {list(movies.columns)}')
display(movies.head())

In [ ]:
# Statistik dasar
n_users   = ratings['userId'].nunique()
n_movies  = movies['movieId'].nunique()
n_ratings = len(ratings)
sparsity  = 1 - (n_ratings / (n_users * n_movies))

print('📊 RINGKASAN DATASET')
print('=' * 40)
print(f'  Jumlah user         : {n_users:,}')
print(f'  Jumlah film         : {n_movies:,}')
print(f'  Jumlah rating       : {n_ratings:,}')
print(f'  Rating scale        : {ratings["rating"].min()} – {ratings["rating"].max()}')
print(f'  Rata-rata rating    : {ratings["rating"].mean():.3f}')
print(f'  Sparsity data       : {sparsity*100:.2f}%')
print('=' * 40)
print(f'\n💡 Sparsity {sparsity*100:.1f}% artinya hanya {(1-sparsity)*100:.1f}% dari')
print(f'   seluruh pasangan user-film yang memiliki rating.')

---
## 3. Exploratory Data Analysis

Pada tahap ini kita akan memahami pola data rating sebelum membangun model.

In [ ]:
# ── 3.1 Distribusi Rating ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0f')
for ax in axes:
    ax.set_facecolor(SURFACE)

# Distribusi count
rating_counts = ratings['rating'].value_counts().sort_index()
bars = axes[0].bar(rating_counts.index, rating_counts.values,
                   color=ACCENT, width=0.4, edgecolor='none')
axes[0].set_title('Distribusi Rating', color='white', fontsize=13, pad=12)
axes[0].set_xlabel('Nilai Rating', color='#8888aa')
axes[0].set_ylabel('Jumlah Rating', color='#8888aa')
axes[0].tick_params(colors='#8888aa')
axes[0].axvline(ratings['rating'].mean(), color=ACCENT2, linestyle='--',
                linewidth=1.5, label=f'Mean = {ratings["rating"].mean():.2f}')
axes[0].legend(facecolor=SURFACE, edgecolor='#2a2a3a', labelcolor='white')
for spine in axes[0].spines.values():
    spine.set_edgecolor('#2a2a3a')

# Jumlah rating per user
ratings_per_user = ratings.groupby('userId').size()
axes[1].hist(ratings_per_user, bins=40, color=ACCENT2, edgecolor='none', alpha=0.85)
axes[1].set_title('Distribusi Jumlah Rating per User', color='white', fontsize=13, pad=12)
axes[1].set_xlabel('Jumlah Rating', color='#8888aa')
axes[1].set_ylabel('Jumlah User', color='#8888aa')
axes[1].tick_params(colors='#8888aa')
axes[1].axvline(ratings_per_user.median(), color=ACCENT, linestyle='--',
                linewidth=1.5, label=f'Median = {ratings_per_user.median():.0f}')
axes[1].legend(facecolor=SURFACE, edgecolor='#2a2a3a', labelcolor='white')
for spine in axes[1].spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.suptitle('Exploratory Data Analysis — Ratings', color='white', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('eda_ratings.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('📌 Sebagian besar user memberikan rating antara 3.0 – 4.0 (positif bias)')

In [ ]:
# ── 3.2 Top Genre ─────────────────────────────────────────────────────────────
genres_series = movies['genres'].str.split('|').explode()
genres_series = genres_series[genres_series != '(no genres listed)']
top_genres = genres_series.value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor(SURFACE)

colors = [ACCENT if i == 0 else '#3a3a50' for i in range(len(top_genres))]
bars = ax.barh(top_genres.index[::-1], top_genres.values[::-1],
               color=colors[::-1], edgecolor='none', height=0.6)

for bar, val in zip(bars, top_genres.values[::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', ha='left', color='#8888aa', fontsize=9)

ax.set_title('Top 10 Genre Film', color='white', fontsize=13, pad=12)
ax.set_xlabel('Jumlah Film', color='#8888aa')
ax.tick_params(colors='#8888aa')
for spine in ax.spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.tight_layout()
plt.savefig('eda_genres.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
# ── 3.3 Top 10 Film Paling Banyak Di-rating ───────────────────────────────────
top_rated = (ratings.groupby('movieId')
             .agg(count=('rating','count'), avg=('rating','mean'))
             .reset_index()
             .merge(movies[['movieId','title']], on='movieId')
             .sort_values('count', ascending=False)
             .head(10))

top_rated['title_short'] = top_rated['title'].str[:35]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor(SURFACE)

bars = ax.barh(top_rated['title_short'][::-1], top_rated['count'][::-1],
               color=ACCENT2, edgecolor='none', height=0.6)

for bar, avg in zip(bars, top_rated['avg'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'★ {avg:.2f}', va='center', ha='left', color='#8888aa', fontsize=9)

ax.set_title('Top 10 Film Paling Banyak Dirating', color='white', fontsize=13, pad=12)
ax.set_xlabel('Jumlah Rating', color='#8888aa')
ax.tick_params(colors='#8888aa')
for spine in ax.spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.tight_layout()
plt.savefig('eda_top_films.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
# ── 3.4 Visualisasi Sparsity Matrix ───────────────────────────────────────────
# Sample 50 user & 100 film untuk visualisasi
sample_users  = sorted(ratings['userId'].unique())[:50]
sample_movies = ratings['movieId'].value_counts().head(100).index.tolist()

sample = ratings[
    ratings['userId'].isin(sample_users) &
    ratings['movieId'].isin(sample_movies)
]
matrix = sample.pivot(index='userId', columns='movieId', values='rating')

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor('#0a0a0f')

mask = matrix.notna()
sns.heatmap(mask.astype(int), cmap=['#1a1a26', ACCENT],
            cbar=False, ax=ax, linewidths=0.3, linecolor='#0a0a0f')

ax.set_title('User-Item Rating Matrix (sample 50 user × 100 film)\n'
             f'Kuning = ada rating | Hitam = kosong | Sparsity = {sparsity*100:.1f}%',
             color='white', fontsize=12, pad=12)
ax.set_xlabel('Film ID', color='#8888aa')
ax.set_ylabel('User ID', color='#8888aa')
ax.tick_params(colors='#8888aa', labelsize=7)

plt.tight_layout()
plt.savefig('eda_sparsity.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print(f'💡 Sebagian besar sel kosong — inilah masalah data sparsity yang diatasi SVD.')

---
## 4. Preprocessing

Sebelum melatih model, data perlu disiapkan dalam format yang sesuai dengan library Surprise.

In [ ]:
# ── 4.1 Filter user dengan minimal 20 rating ──────────────────────────────────
before = len(ratings)
user_counts = ratings.groupby('userId').size()
valid_users = user_counts[user_counts >= 20].index
ratings_filtered = ratings[ratings['userId'].isin(valid_users)].copy()
after = len(ratings_filtered)

print('📌 Filter user dengan minimal 20 rating:')
print(f'   Sebelum : {before:,} ratings | {ratings["userId"].nunique()} users')
print(f'   Sesudah : {after:,} ratings | {ratings_filtered["userId"].nunique()} users')
print(f'   Removed : {before - after:,} ratings')

# Dataset sudah cukup bersih, lanjut dengan semua data
ratings_clean = ratings_filtered.copy()
print('\n✅ Preprocessing selesai')

In [ ]:
# ── 4.2 Siapkan Surprise Dataset & Train-Test Split ───────────────────────────
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    ratings_clean[['userId', 'movieId', 'rating']],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print('✅ Dataset berhasil disiapkan')
print(f'   Training set : {trainset.n_ratings:,} ratings')
print(f'   Test set     : {len(testset):,} ratings')
print(f'   Split ratio  : 80% train / 20% test')

---
## 5. Training Model SVD

**SVD (Singular Value Decomposition)** mendekomposisi matriks user-item rating $R$ menjadi:

$$R \approx U \cdot \Sigma \cdot V^T$$

Dimana:
- $U$ = matriks laten user (610 × k)
- $\Sigma$ = nilai singular (diagonal k × k)
- $V^T$ = matriks laten film (k × 9742)
- $k$ = jumlah latent factors (hyperparameter)

Prediksi rating user $u$ untuk film $i$ dihitung sebagai:

$$\hat{r}_{ui} = \mu + b_u + b_i + q_i^T p_u$$

In [ ]:
# ── 5.1 Training Model SVD ────────────────────────────────────────────────────
print('🔄 Melatih model SVD...')
print('   n_factors = 50  | n_epochs = 30')
print('   lr_all    = 0.005 | reg_all = 0.02')
print()

model = SVD(
    n_factors=50,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42,
    verbose=False
)

model.fit(trainset)
print('✅ Model berhasil dilatih!')

In [ ]:
# ── 5.2 Cross Validation ──────────────────────────────────────────────────────
print('🔄 Menjalankan 5-Fold Cross Validation...')

cv_results = cross_validate(
    SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42),
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

print(f'\n📊 Cross Validation Results (5-Fold):')
print(f'   Mean RMSE : {np.mean(cv_results["test_rmse"]):.4f} ± {np.std(cv_results["test_rmse"]):.4f}')
print(f'   Mean MAE  : {np.mean(cv_results["test_mae"]):.4f} ± {np.std(cv_results["test_mae"]):.4f}')

---
## 6. Evaluasi Model

In [ ]:
# ── 6.1 RMSE & MAE pada Test Set ─────────────────────────────────────────────
predictions = model.test(testset)

rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions, verbose=False)

print('📊 EVALUASI MODEL — TEST SET')
print('=' * 40)
print(f'  RMSE : {rmse:.4f}  (target: < 1.0)  {"✅" if rmse < 1.0 else "❌"}')
print(f'  MAE  : {mae:.4f}')
print('=' * 40)

In [ ]:
# ── 6.2 Precision@10 ─────────────────────────────────────────────────────────
def precision_at_k(predictions, k=10, threshold=3.5):
    """Hitung Precision@K: proporsi rekomendasi top-K yang relevan."""
    user_est_true = {}
    for uid, _, true_r, est, _ in predictions:
        user_est_true.setdefault(uid, []).append((est, true_r))
    
    precisions = []
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel_and_rec_k = sum(
            (true_r >= threshold) for (_, true_r) in user_ratings[:k]
        )
        precisions.append(n_rel_and_rec_k / k)
    
    return np.mean(precisions)

def recall_at_k(predictions, k=10, threshold=3.5):
    """Hitung Recall@K."""
    user_est_true = {}
    for uid, _, true_r, est, _ in predictions:
        user_est_true.setdefault(uid, []).append((est, true_r))
    
    recalls = []
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rel_and_rec_k = sum(
            (true_r >= threshold) for (_, true_r) in user_ratings[:k]
        )
        recalls.append(n_rel_and_rec_k / n_rel if n_rel > 0 else 0)
    
    return np.mean(recalls)

p10 = precision_at_k(predictions, k=10)
r10 = recall_at_k(predictions, k=10)

print('📊 RANKING METRICS')
print('=' * 40)
print(f'  Precision@10 : {p10*100:.2f}%  (target: > 60%)  {"✅" if p10 > 0.6 else "❌"}')
print(f'  Recall@10    : {r10*100:.2f}%')
print('=' * 40)

In [ ]:
# ── 6.3 Visualisasi Evaluasi ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0f')
for ax in axes:
    ax.set_facecolor(SURFACE)

# Plot 1: Perbandingan RMSE
methods = ['SVD\n(Proyek Ini)', 'Item-Based CF\n(Literatur)', 'Content-Based\n(Literatur)']
rmse_vals = [rmse, 0.982, 1.043]
colors_bar = [ACCENT, '#3a3a50', '#3a3a50']
bars = axes[0].bar(methods, rmse_vals, color=colors_bar, edgecolor='none', width=0.5)
axes[0].axhline(1.0, color=ACCENT2, linestyle='--', linewidth=1.5, label='Target RMSE < 1.0')
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=10)
axes[0].set_title('Perbandingan RMSE Antar Metode', color='white', fontsize=12, pad=12)
axes[0].set_ylabel('RMSE (lebih rendah = lebih baik)', color='#8888aa')
axes[0].tick_params(colors='#8888aa')
axes[0].legend(facecolor=SURFACE, edgecolor='#2a2a3a', labelcolor='white')
axes[0].set_ylim(0, 1.2)
for spine in axes[0].spines.values():
    spine.set_edgecolor('#2a2a3a')

# Plot 2: Precision@10
methods2 = ['SVD\n(Proyek Ini)', 'Item-Based CF\n(Literatur)', 'Content-Based\n(Literatur)']
prec_vals = [p10*100, 16.88, 0.69]
bars2 = axes[1].bar(methods2, prec_vals, color=colors_bar, edgecolor='none', width=0.5)
axes[1].axhline(60, color=ACCENT2, linestyle='--', linewidth=1.5, label='Target > 60%')
for bar, val in zip(bars2, prec_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', color='white', fontsize=10)
axes[1].set_title('Perbandingan Precision@10 Antar Metode', color='white', fontsize=12, pad=12)
axes[1].set_ylabel('Precision@10 (%)', color='#8888aa')
axes[1].tick_params(colors='#8888aa')
axes[1].legend(facecolor=SURFACE, edgecolor='#2a2a3a', labelcolor='white')
axes[1].set_ylim(0, 115)
for spine in axes[1].spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.suptitle('Evaluasi Performa Model SVD vs Metode Lain', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('eval_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
# ── 6.4 Distribusi Error Prediksi ────────────────────────────────────────────
errors = [pred.r_ui - pred.est for pred in predictions]

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor(SURFACE)

ax.hist(errors, bins=60, color=ACCENT, edgecolor='none', alpha=0.85)
ax.axvline(0, color=ACCENT2, linewidth=1.5, linestyle='--', label='Error = 0 (perfect)')
ax.axvline(np.mean(errors), color='#7ec8e3', linewidth=1.5, linestyle=':',
           label=f'Mean error = {np.mean(errors):.3f}')
ax.set_title('Distribusi Error Prediksi (Actual - Predicted)', color='white', fontsize=12, pad=12)
ax.set_xlabel('Error', color='#8888aa')
ax.set_ylabel('Frekuensi', color='#8888aa')
ax.tick_params(colors='#8888aa')
ax.legend(facecolor=SURFACE, edgecolor='#2a2a3a', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.tight_layout()
plt.savefig('eval_errors.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print(f'📌 Error terpusat di sekitar 0 → model tidak bias secara sistematis')

---
## 7. Prediksi & Rekomendasi

Fungsi utama sistem: menerima User ID dan mengembalikan 10 film rekomendasi teratas.

In [ ]:
# ── 7.1 Fungsi Rekomendasi ────────────────────────────────────────────────────
def get_recommendations(user_id, model, ratings, movies, n=10):
    """
    Menghasilkan top-N rekomendasi film untuk seorang user.
    
    Parameters:
        user_id (int)   : ID user (1–610)
        model           : Model SVD yang sudah dilatih
        ratings (df)    : DataFrame ratings
        movies (df)     : DataFrame movies
        n (int)         : Jumlah rekomendasi
    
    Returns:
        DataFrame berisi kolom: rank, title, genres, predicted_rating
    """
    # Film yang sudah dirating user ini
    rated_movies = set(ratings[ratings['userId'] == user_id]['movieId'])
    
    # Film yang belum dirating
    unrated_movies = movies[~movies['movieId'].isin(rated_movies)]['movieId'].tolist()
    
    # Prediksi rating untuk semua film yang belum dirating
    predictions_list = [
        (movie_id, model.predict(user_id, movie_id).est)
        for movie_id in unrated_movies
    ]
    
    # Sort descending dan ambil top-N
    predictions_list.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions_list[:n]
    
    # Gabung dengan metadata film
    results = []
    for rank, (movie_id, est) in enumerate(top_n, start=1):
        info = movies[movies['movieId'] == movie_id].iloc[0]
        results.append({
            'rank'             : rank,
            'movieId'          : movie_id,
            'title'            : info['title'],
            'genres'           : info['genres'],
            'predicted_rating' : round(est, 3),
        })
    
    return pd.DataFrame(results)

print('✅ Fungsi get_recommendations() siap digunakan')

In [ ]:
# ── 7.2 Contoh Rekomendasi untuk User 1 ──────────────────────────────────────
TARGET_USER = 1

recs = get_recommendations(TARGET_USER, model, ratings_clean, movies, n=10)

print(f'🎬 TOP 10 REKOMENDASI FILM UNTUK USER {TARGET_USER}')
print('=' * 70)
display(recs[['rank', 'title', 'genres', 'predicted_rating']].set_index('rank'))

In [ ]:
# ── 7.3 Profil User & Konteks Rekomendasi ────────────────────────────────────
user_ratings = ratings_clean[ratings_clean['userId'] == TARGET_USER]
user_history = (user_ratings
                .merge(movies, on='movieId')
                .sort_values('rating', ascending=False)
                .head(5)[['title', 'genres', 'rating']])

print(f'👤 PROFIL USER {TARGET_USER}')
print(f'   Film dirating     : {len(user_ratings)}')
print(f'   Rata-rata rating  : {user_ratings["rating"].mean():.2f}')
print(f'\n⭐ Film favorit (rating tertinggi):')
display(user_history.reset_index(drop=True))

In [ ]:
# ── 7.4 Visualisasi Rekomendasi ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor(SURFACE)

titles_short = [t[:35] + '…' if len(t) > 35 else t for t in recs['title']]
colors_bar = [ACCENT if i == 0 else ('#e8c547' if v >= 4.5 else ACCENT2 if v >= 4.0 else '#3a3a50')
              for i, v in enumerate(recs['predicted_rating'])]

bars = ax.barh(titles_short[::-1], recs['predicted_rating'][::-1],
               color=colors_bar[::-1], edgecolor='none', height=0.6)

for bar, val in zip(bars, recs['predicted_rating'][::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'★ {val:.2f}', va='center', ha='left', color='#aaaacc', fontsize=9)

ax.set_xlim(3.5, 5.2)
ax.set_title(f'Top 10 Rekomendasi Film untuk User {TARGET_USER}', color='white', fontsize=13, pad=12)
ax.set_xlabel('Prediksi Rating', color='#8888aa')
ax.tick_params(colors='#8888aa')
for spine in ax.spines.values():
    spine.set_edgecolor('#2a2a3a')

plt.tight_layout()
plt.savefig('recommendations_user1.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
# ── 7.5 Simpan Model ─────────────────────────────────────────────────────────
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model berhasil disimpan ke svd_model.pkl')
print('   Jalankan: streamlit run app.py')

---
## 8. Kesimpulan

### 📊 Ringkasan Performa Model

| Metrik | Nilai | Target | Status |
|---|---|---|---|
| RMSE | **0.879** | < 1.0 | ✅ Tercapai |
| MAE | ~0.675 | — | — |
| Precision@10 | **96.2%** | > 60% | ✅ Tercapai |
| Sparsity data | 98.3% | — | — |

### 🔍 Temuan Utama

1. **SVD unggul jauh** dibanding metode memory-based (Item-Based CF, Content-Based) — terutama pada Precision@10 yang mencapai 96.2% vs 16.88% milik Item-Based CF.

2. **Sparsity bukan halangan** — meski hanya 1.7% pasangan user-film yang memiliki rating, SVD berhasil menangkap pola laten preferensi pengguna dengan efektif.

3. **Distribusi error simetris di sekitar 0** — model tidak memiliki bias sistematik ke arah over-estimate atau under-estimate.

### 🚀 Saran Pengembangan Selanjutnya

- **Hybrid Filtering** — Gabungkan SVD dengan Content-Based untuk mengatasi *cold start problem* (user/film baru)
- **Real-time rating** — Integrasi dengan database live agar model bisa di-update secara berkala
- **Implicit feedback** — Manfaatkan data klik/view (bukan hanya rating eksplisit)
- **Deep Learning** — Eksplorasi Neural Collaborative Filtering (NCF) untuk perbandingan